# 03 — Export and Validation

Validates the draft EModel/MEModel from the optimisation stage and promotes
them to active lifecycle status.

All validation and plotting settings come from the optimisation recipe
(stored on the TaskResult). The user only needs to provide:
- The optimisation `TaskResult` (from notebook 02)
- The draft `MEModel` entity (registered by notebook 02)

The actual validation runs on **Fargate** via the launch-system.

## Imports

In [ ]:
import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token
from obi_one.core.info import Info
from obi_one.scientific.from_id.memodel_from_id import MEModelFromID
from obi_one.scientific.from_id.task_result_from_id import TaskResultFromID
from obi_one.scientific.tasks.emodel_building.task3_export_and_validation.blocks import (
    ExportAndValidationInitialize,
)

## Connect to entitycore staging

In [ ]:
token = get_token(environment="staging")
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
db_client = Client(
    api_url="https://staging.cell-a.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("Connected to entitycore staging.")

## Build the scan config

Only two inputs are needed:
- `optimization_task_result`: the `TaskResult` from notebook 02
- `memodel`: the draft `MEModel` registered by the optimisation stage

All validation settings (protocols, threshold, plotting) come from the
optimisation recipe stored on the TaskResult.

In [ ]:
# Replace with real entity IDs from your staging project.
OPTIMIZATION_TASK_RESULT_ID = "00000000-0000-0000-0000-000000000001"
MEMODEL_ID = "00000000-0000-0000-0000-000000000002"

scan_config = obi.EModelExportAndValidationScanConfig(
    info=Info(
        campaign_name="L5PC Validation",
        campaign_description="Validate and promote draft L5PC EModel/MEModel to active.",
    ),
    initialize=ExportAndValidationInitialize(
        optimization_task_result=TaskResultFromID(id_str=OPTIMIZATION_TASK_RESULT_ID),
        memodel=MEModelFromID(id_str=MEMODEL_ID),
    ),
)
print(f"Config: {scan_config.campaign_name}")
print(f"TaskResult: {OPTIMIZATION_TASK_RESULT_ID}")
print(f"MEModel: {MEMODEL_ID}")

## Register campaign and config entities

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/03_export_and_validation/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute(db_client=db_client)
print(f"Config entity registered: {grid_scan.single_configs[0].single_entity.id}")

## Launch on Fargate

The validation task runs on Fargate. After completion:
- Draft EModel and MEModel are promoted to `lifecycle_status=active`
- A validation `TaskResult` is registered with figures and details
- `MEModelCalibrationResult` is created with holding/threshold current

In [ ]:
# Launch via the obi-one API (requires the service running):
#
# import httpx
# response = httpx.post(
#     "http://127.0.0.1:8100/declared/task/launch",
#     json={
#         "task_type": "emodel_export_and_validation",
#         "config_id": str(grid_scan.single_configs[0].single_entity.id),
#     },
#     headers={"Authorization": f"Bearer {token}"},
#     timeout=30,
# )
# response.raise_for_status()
# print(response.json())